# ESM HDF5 → JSON Pipeline

This project provides a **robust, restart‑safe pipeline** to download earthquake waveform records from the **European Strong Motion (ESM) database**, store them in **HDF5 format**, and convert them into **JSON files (one per component)**.

The pipeline is designed for **large-scale datasets (thousands of records)** and includes:

- Concurrent downloads
- Batch processing
- Crash‑safe resume capability
- Progress monitoring
- Robust dataframe normalization
- Per‑component JSON output


In [1]:
# Imports / constants

from __future__ import annotations

import csv
import json
import os
import shutil
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Optional, Tuple

import h5py
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

G0 = 9.80665  # m/s^2 per g
HDF5_MAGIC = b"\x89HDF\r\n\x1a\n"


## 0) Robust loader + normalization for `example_df.csv` (2-row header pattern)

In [2]:
def _safe_str(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    return str(x).strip()


def load_example_df_robust(csv_fp: str | Path) -> pd.DataFrame:
    raw = pd.read_csv(csv_fp, header=None)

    cols = [("" if (c is None or (isinstance(c, float) and np.isnan(c))) else str(c)).strip()
            for c in raw.iloc[1].tolist()]

    seen = {}
    fixed = []
    for c in cols:
        base = c if c else "col"
        if base not in seen:
            seen[base] = 0
            fixed.append(base)
        else:
            seen[base] += 1
            fixed.append(f"{base}_{seen[base]}")

    df = raw.iloc[2:].copy()
    df.columns = fixed
    df = df.dropna(axis=1, how="all").reset_index(drop=True)

    for c in df.columns:
        if df[c].dtype == "object":
            df[c] = df[c].astype(str).str.strip()

    df.columns = [str(c).strip() for c in df.columns]
    return df


def normalize_metadata_df(df: pd.DataFrame) -> pd.DataFrame:
    cols = list(df.columns)
    lower = {str(c).lower(): c for c in cols}

    def pick(exact: list[str], contains_any: list[str]) -> str | None:
        for e in exact:
            if e.lower() in lower:
                return lower[e.lower()]
        for c in cols:
            cl = str(c).lower()
            if any(s.lower() in cl for s in contains_any):
                return c
        return None

    event_col = pick(["event_id", "eventid"], ["event"])
    sta_col = pick(["station_code", "station"], ["station"])
    loc_col = pick(["location_code", "location", "station_location"], ["location"])

    if event_col is None or sta_col is None:
        raise ValueError(f"Could not find event/station columns. Available columns: {cols}")

    out = df.copy()
    out = out.rename(columns={event_col: "event_id", sta_col: "station_code"})
    if loc_col is not None:
        out = out.rename(columns={loc_col: "location_code"})
    else:
        out["location_code"] = ""

    for c in ["event_id", "station_code", "location_code"]:
        out[c] = out[c].map(_safe_str)

    out["_invalid_row"] = (out["event_id"] == "") | (out["station_code"] == "")
    return out


## 1) Concurrent downloader

In [3]:
def download_records_to_hdf5(
    df: pd.DataFrame,
    dst_dir: Path,
    *,
    event_id_col: str = "event_id",
    station_code_col: str = "station_code",
    location_code_col: str = "location_code",
    data_type: str = "ACC",
    batch_size: int = 500,
    max_workers: int = 4,
    timeout: Tuple[float, float] = (10.0, 60.0),
    sleep_between: float = 0.1,
    overwrite: bool = False,
    manifest_fp: Optional[Path] = None,
    total_retries: int = 2,
    backoff_factor: float = 0.5,
    show_progress: bool = True,
) -> list[Path]:
    def _ensure_hdf5_file(fp: Path) -> None:
        with open(fp, "rb") as f:
            if f.read(8) != HDF5_MAGIC:
                raise ValueError(f"Downloaded file is not valid HDF5: {fp}")

    def _build_url(event_id: str, station: str) -> str:
        from urllib.parse import urlencode
        qs = urlencode({"eventid": event_id, "station": station, "data-type": data_type})
        return f"https://esm-db.eu/esmws/eventdata/1/query?{qs}"

    def _make_session() -> requests.Session:
        session = requests.Session()
        retry = Retry(
            total=total_retries,
            connect=total_retries,
            read=total_retries,
            status=total_retries,
            backoff_factor=backoff_factor,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=frozenset(["GET"]),
            raise_on_status=False,
        )
        adapter = HTTPAdapter(max_retries=retry, pool_connections=max_workers * 4, pool_maxsize=max_workers * 4)
        session.mount("https://", adapter)
        session.mount("http://", adapter)
        return session

    def _progress_futures(futures, total: int, desc: str):
        if not show_progress:
            for fut in as_completed(futures):
                yield fut
            return
        try:
            from tqdm import tqdm  # type: ignore
            for fut in tqdm(as_completed(futures), total=total, desc=desc):
                yield fut
        except Exception:
            for i, fut in enumerate(as_completed(futures), start=1):
                if i == 1 or i % 50 == 0 or i == total:
                    print(f"{desc}: {i}/{total}")
                yield fut

    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    if manifest_fp is None:
        manifest_fp = dst_dir / "download_manifest.csv"
    manifest_fp = Path(manifest_fp)

    if event_id_col not in df.columns or station_code_col not in df.columns:
        raise ValueError(f"df must contain '{event_id_col}' and '{station_code_col}'")

    df_local = df.copy()
    if location_code_col not in df_local.columns:
        df_local[location_code_col] = ""

    for c in [event_id_col, station_code_col, location_code_col]:
        df_local[c] = df_local[c].map(_safe_str)

    df_local["record_id"] = df_local[event_id_col] + "__" + df_local[station_code_col] + "__" + df_local[location_code_col]

    manifest_cols = ["record_id","event_id","station_code","location_code","url","status","http_status","error_type","error_message","timestamp"]
    if not manifest_fp.exists():
        with open(manifest_fp, "w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=manifest_cols).writeheader()

    done: set[str] = set()
    if manifest_fp.exists() and not overwrite:
        try:
            m = pd.read_csv(manifest_fp, dtype=str).fillna("")
            done = set(m.loc[m["status"] == "downloaded", "record_id"].tolist())
        except Exception:
            done = set()

    session = _make_session()
    ok_files: list[Path] = []

    def _download_one(row_dict: dict[str, str]) -> dict[str, Any]:
        rec_id = row_dict["record_id"]
        event_id = row_dict[event_id_col]
        station_code = row_dict[station_code_col]
        loc_code = row_dict[location_code_col]
        dst_fp = dst_dir / f"{rec_id}.h5"

        if not event_id or not station_code:
            return {"record_id": rec_id,"event_id": event_id,"station_code": station_code,"location_code": loc_code,
                    "url": "", "status": "invalid_metadata","http_status":"","error_type":"ValueError",
                    "error_message":"Missing event_id or station_code; request skipped.","timestamp": pd.Timestamp.now("UTC").isoformat()}

        if not overwrite and (rec_id in done or dst_fp.exists()):
            return {"record_id": rec_id,"event_id": event_id,"station_code": station_code,"location_code": loc_code,
                    "url": "", "status": "skipped","http_status":"","error_type":"","error_message":"",
                    "timestamp": pd.Timestamp.now("UTC").isoformat()}

        url = _build_url(event_id, station_code)
        status, http_status, err_type, err_msg = "failed", "", "", ""

        try:
            with session.get(url, stream=True, timeout=timeout) as r:
                http_status = str(r.status_code)
                if r.status_code != 200:
                    raise requests.HTTPError(f"HTTP {r.status_code} for {url}")

                tmp = dst_fp.with_suffix(f".part.{os.getpid()}")
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
                tmp.replace(dst_fp)

            _ensure_hdf5_file(dst_fp)
            status = "downloaded"
        except Exception as e:
            err_type = type(e).__name__
            err_msg = str(e)[:500]
            for p in dst_dir.glob(dst_fp.stem + ".part*"):
                try: p.unlink()
                except Exception: pass

        time.sleep(sleep_between)
        return {"record_id": rec_id,"event_id": event_id,"station_code": station_code,"location_code": loc_code,
                "url": url,"status": status,"http_status": http_status,"error_type": err_type,"error_message": err_msg,
                "timestamp": pd.Timestamp.now("UTC").isoformat()}

    n = len(df_local)
    for bstart in range(0, n, batch_size):
        bend = min(bstart + batch_size, n)
        batch = df_local.iloc[bstart:bend]
        if not overwrite and done:
            batch = batch[~batch["record_id"].isin(done)]
        rows = batch.to_dict(orient="records")
        if not rows:
            continue

        print(f"Batch {bstart+1}-{bend} (submitting {len(rows)} tasks) ...")

        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            futures = [ex.submit(_download_one, r) for r in rows]
            with open(manifest_fp, "a", newline="", encoding="utf-8") as f:
                w = csv.DictWriter(f, fieldnames=manifest_cols)
                for fut in _progress_futures(futures, total=len(futures), desc="Downloading"):
                    out = fut.result()
                    w.writerow(out); f.flush()
                    if out["status"] == "downloaded":
                        done.add(out["record_id"])
                        ok_files.append(dst_dir / f"{out['record_id']}.h5")

    return ok_files


## 2) HDF5 → per-component JSON

In [ ]:
def hdf5_record_to_json(
    hdf5_fp: Path,
    dst_dir: Path,
    rec_component: list[str] = ["U", "V", "W"],
    *,
    metadata: Optional[dict[str, Any]] = None,
    assume_input_units: str = "m/s^2",
    event_id_col: str = "event_id",
    station_code_col: str = "station_code",
    location_code_col: str = "location_code",
) -> list[Path]:

    def _to_g(values: np.ndarray, units: str) -> np.ndarray:
        u = (units or "").strip().lower()
        if u in ("g", "grav", "gravity", "gravities"):
            return values.astype(float)
        if "m/s" in u:
            return values.astype(float) / G0
        if "cm/s" in u or "gal" in u:
            return values.astype(float) / (100.0 * G0)
        return values.astype(float)

    def _find_station_group(h5: h5py.File, station_code: str) -> h5py.Group:
        if "Waveforms" not in h5:
            raise KeyError("HDF5 missing /Waveforms")
        wf = h5["Waveforms"]

        # If single group, use it
        if len(wf.keys()) == 1:
            return wf[list(wf.keys())[0]]

        # Else, match station_code anywhere in the key (robust)
        matches = []
        for k in wf.keys():
            ks = str(k)
            if f".{station_code}" in ks or ks.endswith(station_code):
                matches.append(k)

        if len(matches) == 1:
            return wf[matches[0]]
        if len(matches) > 1:
            # choose shortest / most specific
            matches = sorted(matches, key=lambda x: len(str(x)))
            return wf[matches[0]]

        raise KeyError(f"Station group not found for station_code={station_code}. Waveforms keys={list(wf.keys())[:10]}")

    meta = dict(metadata or {})
    event_id = _safe_str(meta.get(event_id_col))
    station_code = _safe_str(meta.get(station_code_col))
    location_code = _safe_str(meta.get(location_code_col))

    if not event_id or not station_code:
        raise ValueError(f"metadata must include '{event_id_col}' and '{station_code_col}'")

    base_id = "__".join([event_id, station_code, location_code])
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    written: list[Path] = []

    with h5py.File(Path(hdf5_fp), "r") as h5:
        stgrp = _find_station_group(h5, station_code=station_code)

        # FIX: take all datasets in station group
        ds_list: list[h5py.Dataset] = []
        for k in sorted(stgrp.keys()):
            obj = stgrp[k]
            if isinstance(obj, h5py.Dataset):
                ds_list.append(obj)

        if not ds_list:
            raise KeyError("No datasets found in station group")

        if len(ds_list) < len(rec_component):
            raise KeyError(f"Found only {len(ds_list)} datasets, need {len(rec_component)} for {rec_component}")

        selected = ds_list[: len(rec_component)]

        for comp, ds in zip(rec_component, selected):
            arr = np.array(ds[()], dtype=float).ravel()
            arr_g = _to_g(arr, assume_input_units)

            sr = ds.attrs.get("sampling_rate", None)
            if sr is None:
                raise KeyError(f"sampling_rate missing for {ds.name}")
            dt = 1.0 / float(sr)

            out = {
                "eq_index": base_id,
                "database": "esm",
                "dt": float(dt),
                "units": "g",
                "record_type": "acc",
                "component": comp,
                "record": arr_g.tolist(),
            }
            for k, v in meta.items():
                if k not in out:
                    out[k] = v

            out_fp = dst_dir / f"{base_id}_{comp}.json"
            with open(out_fp, "w", encoding="utf-8") as f:
                json.dump(out, f, indent=2, ensure_ascii=False)
            written.append(out_fp)

    return written


## 3) Copy subset JSONs

In [5]:
def copy_records(
    df_subset: pd.DataFrame,
    src_json_dir: Path,
    dst_json_dir: Path,
    *,
    rec_component: list[str] = ["U", "V", "W"],
    event_id_col: str = "event_id",
    station_code_col: str = "station_code",
    location_code_col: str = "location_code",
    overwrite: bool = False,
) -> list[Path]:
    src_json_dir = Path(src_json_dir)
    dst_json_dir = Path(dst_json_dir)
    dst_json_dir.mkdir(parents=True, exist_ok=True)

    df_local = df_subset.copy()
    if location_code_col not in df_local.columns:
        df_local[location_code_col] = ""
    for c in [event_id_col, station_code_col, location_code_col]:
        df_local[c] = df_local[c].map(_safe_str)

    copied: list[Path] = []
    for _, row in df_local.iterrows():
        base_id = "__".join([row[event_id_col], row[station_code_col], row[location_code_col]])
        for comp in rec_component:
            src_fp = src_json_dir / f"{base_id}_{comp}.json"
            if not src_fp.exists():
                continue
            dst_fp = dst_json_dir / src_fp.name
            if dst_fp.exists() and not overwrite:
                copied.append(dst_fp)
                continue
            shutil.copy2(src_fp, dst_fp)
            copied.append(dst_fp)

    return copied


## Run the pipeline

In [6]:
# ---- Pipeline runner ----

CSV_PATH = Path("example_df.csv")  # change to your real metadata CSV path

HDF5_DIR = Path("out_hdf5")
JSON_DIR = Path("out_json")
JSON_SUBSET_DIR = Path("out_json_subset")

# Load + normalize
df_raw = load_example_df_robust(CSV_PATH)
df_norm = normalize_metadata_df(df_raw)

invalid = df_norm[df_norm["_invalid_row"]].copy()
if len(invalid) > 0:
    print(f"⚠️ Found {len(invalid)} invalid rows (missing event_id or station_code). They will be skipped for download.")

df = df_norm[~df_norm["_invalid_row"]].drop(columns=["_invalid_row"]).copy()
df = df.drop_duplicates(subset=["event_id", "station_code", "location_code"]).reset_index(drop=True)

print("Rows to process:", len(df))
display(df.head())

# Download (resume-safe)
downloaded = download_records_to_hdf5(
    df=df,
    dst_dir=HDF5_DIR,
    batch_size=500,
    max_workers=4,
    timeout=(10.0, 60.0),
    sleep_between=0.1,
    overwrite=False,
    manifest_fp=HDF5_DIR / "download_manifest.csv",
    total_retries=2,
    backoff_factor=0.5,
    show_progress=True,
)
print("HDF5 downloaded/present:", len(downloaded))

# Convert
JSON_DIR.mkdir(parents=True, exist_ok=True)
written = 0
skipped_missing = 0
conv_errors = []

try:
    from tqdm import tqdm  # type: ignore
    iterator = tqdm(df.itertuples(index=False), total=len(df), desc="Converting")
except Exception:
    iterator = df.itertuples(index=False)

for row in iterator:
    base_id = f"{row.event_id}__{row.station_code}__{row.location_code}"
    h5_fp = HDF5_DIR / f"{base_id}.h5"
    if not h5_fp.exists():
        skipped_missing += 1
        continue
    try:
        files = hdf5_record_to_json(
            hdf5_fp=h5_fp,
            dst_dir=JSON_DIR,
            rec_component=["U", "V", "W"],
            metadata=row._asdict(),
            assume_input_units="m/s^2",
        )
        written += len(files)
    except Exception as e:
        conv_errors.append({"record_id": base_id, "error": repr(e)})

print("JSON files written:", written)
print("Skipped (missing h5):", skipped_missing)

if conv_errors:
    err_fp = JSON_DIR / "conversion_errors.csv"
    pd.DataFrame(conv_errors).to_csv(err_fp, index=False)
    print(f"⚠️ Conversion errors: {len(conv_errors)}. Logged to: {err_fp.resolve()}")

# Copy subset
subset = df.head(100).copy()
copied = copy_records(subset, JSON_DIR, JSON_SUBSET_DIR, overwrite=False)
print("Copied subset JSON files:", len(copied), "->", JSON_SUBSET_DIR.resolve())


Rows to process: 20


,col,index,event_id,event_time,station_code,location_code,trt,mag,rjb,vs30,...,SA(0.521)_1,SA(0.705)_1,SA(0.955)_1,SA(1.294)_1,SA(1.753)_1,SA(2.375)_1,SA(3.218)_1,SA(4.359)_1,SA(5.905)_1,SA(8.0)_1
0,7326.0,8308,EMSC-20160824_0000006,2016-08-24 01:36:32,AQG,00,Shallow Default,6.0,29.52,696.0,...,0.17060196776902423,0.09382644536402351,0.06814023544811426,0.039502568845574425,0.030294699706746166,0.02021049393423339,0.009898052231336758,0.0074626791381576935,0.003720099472002657,0.0018639177309363137
1,9591.0,10632,EMSC-20161026_0000095,2016-10-26 19:18:06,MMO,00,Shallow Default,5.9,10.4,787.333015,...,0.12519842268648415,0.10933148358618745,0.07268812915374484,0.04671992861720903,0.026877792494537443,0.014136594897238357,0.006729307557097281,0.004070654828503528,0.0025750994937274014,0.001355383646091483
2,28717.0,8407,EMSC-20160824_0000006,2016-08-24 01:36:32,SPD,00,Shallow Default,6.0,16.12,968.233606,...,0.1528336544971788,0.2097170338686229,0.07997722644920122,0.04477769771442282,0.028209681415668268,0.013020642708425885,0.011508384344045504,0.007316861148180679,0.0034399222559371116,0.001490516322136512
3,35447.0,15810,IT-2009-0102,2009-04-07 17:47:37,RM13,00,Shallow Default,5.54,12.29,792.446158,...,0.19585236773857367,0.14291795588270556,0.07461652660389174,0.0423308165302268,0.018724608875310397,0.010680127071733082,0.006190656076177777,0.004047564164131357,0.002256867307161825,0.001194362134263668
4,7405.0,8387,EMSC-20160824_0000006,2016-08-24 01:36:32,PZI1,00,Shallow Default,6.0,22.73,855.838905,...,0.10843947925862688,0.069798872324838,0.060368279810989335,0.055120878653292724,0.03494591444600918,0.019198534318406787,0.013850729101803731,0.007473022098915529,0.004283117678601865,0.001466117646883357


Batch 1-20 (submitting 20 tasks) ...


Downloading: 100%|██████████| 20/20 [01:56<00:00,  5.85s/it]


HDF5 downloaded/present: 17


Converting: 100%|██████████| 20/20 [00:01<00:00, 10.65it/s]


JSON files written: 51
Skipped (missing h5): 3
Copied subset JSON files: 51 -> C:\Users\Jan-Matthis Deda\Desktop\NIC\hdf5\project\final\out_json_subset
